In [1]:
import os
import pandas as pd
import numpy as np

BASE = "/kaggle/input/datasets/aadityaupadhyay1353"
OUT = "/kaggle/working/"

os.makedirs(OUT, exist_ok=True)

print("Statistical analysis notebook started")

Statistical analysis notebook started


In [2]:
baseline = pd.read_csv(
    f"{BASE}/v2-baseline-results/v2_model_baseline_results.csv"
)

print("Baseline results:")
display(baseline)

Baseline results:


,window,model,precision,recall,f1
0,20,random_forest,0.873377,0.853968,0.863563
1,30,random_forest,0.869469,0.845161,0.857143
2,30,logistic,0.709321,0.965591,0.817851
3,10,random_forest,0.839744,0.793939,0.816199
4,20,xgboost,0.708235,0.955556,0.813514
5,30,xgboost,0.701954,0.926882,0.798888
6,10,xgboost,0.663866,0.957576,0.784119
7,20,logistic,0.649789,0.977778,0.780735
8,10,logistic,0.578947,1.000000,0.733333


In [3]:
aggregation = pd.read_csv(
    f"{BASE}/v2-aggregation-results/v2_aggregation_comparison.csv"
)

print("Aggregation strategies:")
display(aggregation)

Aggregation strategies:


,strategy,precision,recall,f1
0,Majority,0.80618,0.911111,0.855440
1,Weighted-F1,0.80618,0.911111,0.855440
2,Risk-Based,0.60198,0.965079,0.741463


In [4]:
robust_dist = pd.read_csv(
    f"{BASE}/v2-robustness-results/v2_robustness_distributed.csv"
)

robust_cent = pd.read_csv(
    f"{BASE}/v2-robustness-results/v2_robustness_centralized.csv"
)

robust_delta = pd.read_csv(
    f"{BASE}/v2-robustness-results/v2_robustness_delta.csv"
)

print("Robustness results loaded")

Robustness results loaded


In [5]:
best_baseline = baseline.sort_values("f1", ascending=False).iloc[0]

print("Best centralized model:")
print(best_baseline)

Best centralized model:
window                  20
model        random_forest
precision         0.873377
recall            0.853968
f1                0.863563
Name: 0, dtype: object


In [6]:
best_agg = aggregation.sort_values("f1", ascending=False).iloc[0]

print("Best aggregation strategy:")
print(best_agg)

Best aggregation strategy:
strategy     Majority
precision     0.80618
recall       0.911111
f1            0.85544
Name: 0, dtype: object


In [7]:
comparison_rows = []

comparison_rows.append({
    "model_type": "Centralized Baseline",
    "precision": best_baseline["precision"],
    "recall": best_baseline["recall"],
    "f1": best_baseline["f1"]
})

comparison_rows.append({
    "model_type": "Distributed CPS",
    "precision": best_agg["precision"],
    "recall": best_agg["recall"],
    "f1": best_agg["f1"]
})

comparison_df = pd.DataFrame(comparison_rows)

comparison_df

,model_type,precision,recall,f1
0,Centralized Baseline,0.873377,0.853968,0.863563
1,Distributed CPS,0.806180,0.911111,0.855440


In [8]:
comparison_df.to_csv(
    f"{OUT}/model_comparison_table.csv",
    index=False
)

print("Saved model comparison table")

Saved model comparison table


In [9]:
summary = robust_delta.groupby("experiment").agg({

    "f1_dist":"mean",
    "f1_cent":"mean",
    "f1_delta":"mean"

}).reset_index()

summary

,experiment,f1_dist,f1_cent,f1_delta
0,missing_values,0.856433,0.831424,0.025009
1,node_dropout,0.858734,0.190338,0.668396
2,sensor_failure,0.853756,0.793998,0.059758
3,sensor_noise,0.853017,0.841115,0.011902


In [10]:
summary.to_csv(
    f"{OUT}/robustness_summary.csv",
    index=False
)

print("Saved robustness summary")

Saved robustness summary


In [11]:
ranking = robust_delta.groupby("experiment")["f1_delta"].mean().sort_values(ascending=False).reset_index()

ranking

,experiment,f1_delta
0,node_dropout,0.668396
1,sensor_failure,0.059758
2,missing_values,0.025009
3,sensor_noise,0.011902


In [12]:
ranking.to_csv(
    f"{OUT}/robustness_ranked.csv",
    index=False
)

print("Saved robustness ranking")

Saved robustness ranking


In [13]:
!ls -lh /kaggle/working/

total 40K
-rw-r--r-- 1 root root 182 Apr 12 14:22 model_comparison_table.csv
---------- 1 root root 26K Apr 12 14:22 __notebook__.ipynb
-rw-r--r-- 1 root root 157 Apr 12 14:22 robustness_ranked.csv
-rw-r--r-- 1 root root 326 Apr 12 14:22 robustness_summary.csv
